In [1]:
import random

# The eight ways to win: three rows, three columns, two diagonals,
# written as positions on a board numbered 0-8 left to right, top to bottom.
WINS = [(0,1,2),(3,4,5),(6,7,8),(0,3,6),(1,4,7),(2,5,8),(0,4,8),(2,4,6)]

def winner(board):
    """Return 'X', 'O', 'draw', or None if the game is still going."""
    for a, b, c in WINS:
        if board[a] != '.' and board[a] == board[b] == board[c]:
            return board[a]
    return 'draw' if '.' not in board else None

def moves(board):
    """Every empty square, as a list of positions."""
    return [i for i, c in enumerate(board) if c == '.']

def play(board, pos, mark):
    """Return a NEW board with mark placed at pos. Boards are never edited in place."""
    return board[:pos] + mark + board[pos+1:]

def show(board):
    for r in range(0, 9, 3):
        print(' ' + ' | '.join(board[r:r+3]))

In [2]:
ALPHA   = 0.3     # Keeps updates gradual so noisy random games don't overwrite good estimates
GAMMA   = 0.9     # Values future wins almost as much as immediate ones so winning paths propagate back
EPSILON = 0.2     # Forces 20% random moves during training so the agent discovers new board states

REWARD_WIN  =  1.0   # Positive target so the agent actively seeks winning lines
REWARD_LOSS = -1.0   # Negative penalty so the agent learns to block instead of treating losses as neutral
REWARD_DRAW =  0.2   # Small positive reward so a draw is better than losing, but still worse than winning

Q = {}            # board string -> {position: value}

def qrow(board):
    """Get this board's row of the Q-table, creating it the first time we see it."""
    if board not in Q:
        Q[board] = {m: 0.0 for m in moves(board)}
    return Q[board]

def choose(board, rng):
    """Pick a move: explore at random, or take the best one we know about."""
    row = qrow(board)
    if rng.random() < EPSILON:
        return rng.choice(moves(board))
    return max(row, key=row.get)

def update(board, action, reward, next_board, rng):
    """Apply the Q-learning rule. This is the same line you wrote in Assignment 6."""

    # If the game ended there is no next position, so future value is 0.
    # Otherwise it is the best move available from where we landed.
    if next_board is None:
        future = 0.0
    else:
        future = max(qrow(next_board).values())

    # Nudge the current move's estimate toward the immediate reward plus the discounted
    # best future move so credit for wins and losses flows backwards to earlier turns
    Q[board][action] = Q[board][action] + ALPHA * (reward + GAMMA * future - Q[board][action])

In [3]:
def evaluate(n_games=1000, seed=42):
    """Play n_games with exploration turned off and return (win%, draw%, loss%)."""
    rng = random.Random(seed)
    wins = draws = losses = 0

    for _ in range(n_games):
        board = '.........'
        while True:
            # Break ties randomly so an empty Q-table plays truly at random for the baseline
            if board not in Q:
                action = rng.choice(moves(board))
            else:
                row = Q[board]
                best_val = max(row.values())
                best_moves = [m for m, v in row.items() if v == best_val]
                action = rng.choice(best_moves)

            board = play(board, action, 'X')
            res = winner(board)
            if res == 'X':
                wins += 1
                break
            elif res == 'draw':
                draws += 1
                break

            # Random opponent O
            board = play(board, rng.choice(moves(board)), 'O')
            res = winner(board)
            if res == 'O':
                losses += 1
                break
            elif res == 'draw':
                draws += 1
                break

    return (wins / n_games * 100, draws / n_games * 100, losses / n_games * 100)

# 3.1 Baseline before any training
w, d, l = evaluate()
print(f"Baseline (0 games): Win={w:.1f}% | Draw={d:.1f}% | Loss={l:.1f}%\n")

# 3.2 Train for 50,000 games, evaluating every 5,000
train_rng = random.Random(7)
print("| Episodes | Win % | Draw % | Loss % |")
print("|----------|-------|--------|--------|")
print(f"| 0        | {w:5.1f} | {d:6.1f} | {l:6.1f} |")

for ep in range(1, 50001):
    board = '.........'
    while True:
        action = choose(board, train_rng)
        after_x = play(board, action, 'X')
        res = winner(after_x)

        if res == 'X':
            update(board, action, REWARD_WIN, None, train_rng)
            break
        elif res == 'draw':
            update(board, action, REWARD_DRAW, None, train_rng)
            break

        # Random opponent O responds before X's turn comes back around
        after_o = play(after_x, train_rng.choice(moves(after_x)), 'O')
        res = winner(after_o)

        if res == 'O':
            update(board, action, REWARD_LOSS, None, train_rng)
            break
        elif res == 'draw':
            update(board, action, REWARD_DRAW, None, train_rng)
            break

        # Game continues: pass 0 immediate reward and link to the next board X sees
        update(board, action, 0.0, after_o, train_rng)
        board = after_o

    if ep % 5000 == 0:
        w, d, l = evaluate()
        print(f"| {ep:<8} | {w:5.1f} | {d:6.1f} | {l:6.1f} |")

# 4.2 Count how many unique board states the agent actually visited
print(f"\nPositions in the Q-table: {len(Q):,}")

Baseline (0 games): Win=59.6% | Draw=13.2% | Loss=27.2%

| Episodes | Win % | Draw % | Loss % |
|----------|-------|--------|--------|
| 0        |  59.6 |   13.2 |   27.2 |
| 5000     |  94.4 |    4.3 |    1.3 |
| 10000    |  98.4 |    1.6 |    0.0 |
| 15000    |  99.6 |    0.4 |    0.0 |
| 20000    |  98.5 |    0.7 |    0.8 |
| 25000    |  98.9 |    1.1 |    0.0 |
| 30000    |  99.4 |    0.6 |    0.0 |
| 35000    |  98.4 |    1.6 |    0.0 |
| 40000    |  99.3 |    0.7 |    0.0 |
| 45000    |  99.6 |    0.2 |    0.2 |
| 50000    |  99.9 |    0.1 |    0.0 |

Positions in the Q-table: 1,832


In [ ]:
def play_human(games=5):
    """Let a human play as O against the trained X agent (exploration off)."""
    print("Board positions:\n 0 | 1 | 2\n 3 | 4 | 5\n 6 | 7 | 8\n")
    results = {'X': 0, 'O': 0, 'draw': 0}

    for g in range(1, games + 1):
        print(f"=== Game {g} of {games} ===")
        board = '.........'
        while True:
            # Agent (X) plays greedily using what it learned
            if board in Q:
                action = max(Q[board], key=Q[board].get)
            else:
                action = moves(board)[0]
            board = play(board, action, 'X')
            show(board)

            res = winner(board)
            if res is not None:
                print(f"Result: {res}\n")
                results[res] += 1
                break

            # Human (O) move
            valid = moves(board)
            while True:
                try:
                    pos = int(input(f"Your move {valid}: "))
                    if pos in valid:
                        break
                except ValueError:
                    pass
                print("Pick an empty square number from the list.")

            board = play(board, pos, 'O')
            res = winner(board)
            if res is not None:
                show(board)
                print(f"Result: {res}\n")
                results[res] += 1
                break

    print(f"Final Score — Agent (X): {results['X']} | You (O): {results['O']} | Draws: {results['draw']}")

play_human(5)

Board positions:
 0 | 1 | 2
 3 | 4 | 5
 6 | 7 | 8

=== Game 1 of 5 ===
 X | . | .
 . | . | .
 . | . | .
Your move [1, 2, 3, 4, 5, 6, 7, 8]: 1
 X | O | .
 X | . | .
 . | . | .
Your move [2, 4, 5, 6, 7, 8]: 3
Pick an empty square number from the list.
Your move [2, 4, 5, 6, 7, 8]: 4
 X | O | .
 X | O | .
 X | . | .
Result: X

=== Game 2 of 5 ===
 X | . | .
 . | . | .
 . | . | .
Your move [1, 2, 3, 4, 5, 6, 7, 8]: 4
 X | . | .
 X | O | .
 . | . | .
Your move [1, 2, 5, 6, 7, 8]: 6
 X | . | X
 X | O | .
 O | . | .
Your move [1, 5, 7, 8]: 1
 X | O | X
 X | O | .
 O | X | .
Your move [5, 8]: 5
 X | O | X
 X | O | O
 O | X | X
Result: draw

=== Game 3 of 5 ===
 X | . | .
 . | . | .
 . | . | .
